#### Create Spark Session

In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    #.set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.0.0")
    #.set("spark.executor.heartbeatInterval", "300000")
    #.set("spark.network.timeout", "400000")
    #.set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    #.set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    #.set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    #.set("spark.hadoop.fs.s3a.path.style.access", "true")
    #.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    #.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    #.set("spark.sql.warehouse.dir", "s3a://defaultfs/spark/warehouse")
    #.set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

spark = (
    SparkSession.builder.master("local[*]").
        appName('spark-joins-notebook').
        config(conf=sparkConf).
        getOrCreate()
)

spark.sparkContext.setLogLevel('ERROR')
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/30 17:08:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Spark Configurations

#### Set

In [2]:
spark.conf.set("spark.sql.shuffle.partitions", "8")

#### Get

In [3]:
print(spark.conf.get("spark.sql.shuffle.partitions"))

8


### Read Data

In [2]:
payment_col= ["paymentId", "customerId","amount"]

payment_data = [
    (1, 101, 250), 
    (2, 102, 111), 
    (3, 103, 500), 
    (4, 104, 400), 
    (5, 105, 150), 
    (6, 106, 450)
]

payment_df = spark.createDataFrame(payment_data,payment_col)
payment_df.show()

customer_col = ["customerId", "name"]

customer_data = [
    (101,"Jon"), 
    (102,"Aron"),
    (103,"Sam")
]

customer_df = spark.createDataFrame(customer_data,customer_col)
customer_df.show()

+---------+----------+------+
|paymentId|customerId|amount|
+---------+----------+------+
|        1|       101|   250|
|        2|       102|   111|
|        3|       103|   500|
|        4|       104|   400|
|        5|       105|   150|
|        6|       106|   450|
+---------+----------+------+

+----------+----+
|customerId|name|
+----------+----+
|       101| Jon|
|       102|Aron|
|       103| Sam|
+----------+----+



In [12]:
#
# Inner join
#
cust_patments = customer_df.join(payment_df, customer_df["customerId"] == payment_df["customerId"], "inner" )
cust_patments.show()

+----------+----+---------+----------+------+
|customerId|name|paymentId|customerId|amount|
+----------+----+---------+----------+------+
|       101| Jon|        1|       101|   250|
|       102|Aron|        2|       102|   111|
|       103| Sam|        3|       103|   500|
+----------+----+---------+----------+------+



In [13]:
#
# Left join
#
leftJoinDf = payment_df.join(customer_df, customer_df["customerId"] == payment_df["customerId"], "left" )
leftJoinDf.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|   250|       101| Jon|
|        2|       102|   111|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|       105|   150|      NULL|NULL|
|        6|       106|   450|      NULL|NULL|
+---------+----------+------+----------+----+



In [6]:
#
# Right join
#
rightJoinDf = payment_df.join(customer_df, customer_df["customerId"] == payment_df["customerId"], "right" )
rightJoinDf.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|   250|       101| Jon|
|        2|       102|   111|       102|Aron|
|        3|       103|   500|       103| Sam|
+---------+----------+------+----------+----+



In [7]:
#
# Right join
#
fullJoinDf = payment_df.join(customer_df, customer_df["customerId"] == payment_df["customerId"], "outer")
fullJoinDf.show()

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|        1|       101|   250|       101| Jon|
|        2|       102|   111|       102|Aron|
|        3|       103|   500|       103| Sam|
|        4|       104|   400|      NULL|NULL|
|        5|       105|   150|      NULL|NULL|
|        6|       106|   450|      NULL|NULL|
+---------+----------+------+----------+----+



In [8]:
#
# crossJoin
#

spark.conf.set("spark.sql.crossJoin.enabled", True)

#
payment_df.crossJoin(customer_df).show(truncate=False)

+---------+----------+------+----------+----+
|paymentId|customerId|amount|customerId|name|
+---------+----------+------+----------+----+
|1        |101       |250   |101       |Jon |
|1        |101       |250   |102       |Aron|
|1        |101       |250   |103       |Sam |
|2        |102       |111   |101       |Jon |
|2        |102       |111   |102       |Aron|
|2        |102       |111   |103       |Sam |
|3        |103       |500   |101       |Jon |
|3        |103       |500   |102       |Aron|
|3        |103       |500   |103       |Sam |
|4        |104       |400   |101       |Jon |
|4        |104       |400   |102       |Aron|
|4        |104       |400   |103       |Sam |
|5        |105       |150   |101       |Jon |
|5        |105       |150   |102       |Aron|
|5        |105       |150   |103       |Sam |
|6        |106       |450   |101       |Jon |
|6        |106       |450   |102       |Aron|
|6        |106       |450   |103       |Sam |
+---------+----------+------+-----

In [9]:
#
# Left Anti : that returns only the rows from the left dataframe that do not have matching values in the right dataframe. It is used to find the rows in one dataframe that do not have corresponding values in another dataframe.
#

payment_df.join(customer_df, payment_df["customerId"] == customer_df["customerId"], "left_anti").show(truncate=False)

+---------+----------+------+
|paymentId|customerId|amount|
+---------+----------+------+
|4        |104       |400   |
|5        |105       |150   |
|6        |106       |450   |
+---------+----------+------+



##### `Left Semi` : returns only the columns from the left dataframe that have matching values in the right dataframe. It is used to find the values in one dataframe that have corresponding values in another dataframe.

In [10]:
#
# Left Semi : returns only the columns from the left dataframe that have matching values in the right dataframe. It is used to find the values in one dataframe that have corresponding values in another dataframe.
#

payment_df.join(customer_df, payment_df["customerId"] == customer_df["customerId"],"leftsemi").show(truncate=False)

+---------+----------+------+
|paymentId|customerId|amount|
+---------+----------+------+
|1        |101       |250   |
|2        |102       |111   |
|3        |103       |500   |
+---------+----------+------+



In [ ]:
#
# Self Join :
#

payment_df.alias("payment_df1").join(payment_df.alias("payment_df2"), payment_df['customerId'] == payment_df['customerId']).show()

+---------+----------+------+---------+----------+------+
|paymentId|customerId|amount|paymentId|customerId|amount|
+---------+----------+------+---------+----------+------+
|        1|       101|  2500|        1|       101|  2500|
|        2|       102|  1110|        2|       102|  1110|
|        3|       103|   500|        3|       103|   500|
|        4|       104|   400|        4|       104|   400|
|        5|       105|   150|        5|       105|   150|
|        6|       106|   450|        6|       106|   450|
+---------+----------+------+---------+----------+------+



In [5]:

employee_columns = ['emp_id', 'emp_name', 'emp_role', 'emp_manager', 'emp_hiredate', 'emp_salary', 'emp_comm', 'emp_dept']

employee_schema = StructType() \
    .add("emp_id", IntegerType(), True) \
    .add("emp_name", StringType(), True) \
    .add("emp_role", StringType(), True) \
    .add("emp_manager", StringType(), True) \
    .add("emp_hiredate", DateType(), True) \
    .add("emp_salary", IntegerType(), True) \
    .add("emp_comm", IntegerType(), True) \
    .add("emp_dept", IntegerType(), True)

employee_df = spark.read.csv("file:///apps/sandbox/datasets/employee-data/employee.csv",
    header=True,
    schema=employee_schema
)

#
# employee_df.printSchema()

#
# print(employee_df.rdd.getNumPartitions())

#
employee_df.show(truncate=False)

+------+--------+---------+-----------+------------+----------+--------+--------+
|emp_id|emp_name|emp_role |emp_manager|emp_hiredate|emp_salary|emp_comm|emp_dept|
+------+--------+---------+-----------+------------+----------+--------+--------+
|7839  |KING    |PRESIDENT|NULL       |1981-11-17  |5000      |NULL    |10      |
|7698  |BLAKE   |MANAGER  |7839       |1981-01-05  |2850      |125     |30      |
|7782  |CLARK   |MANAGER  |7839       |1983-09-06  |2450      |NULL    |10      |
|7566  |JONES   |MANAGER  |7839       |1981-02-04  |2975      |345     |20      |
|7788  |SCOTT   |ANALYST  |7566       |1984-12-17  |3000      |NULL    |20      |
|7902  |FORD    |ANALYST  |7566       |1981-11-27  |3000      |NULL    |20      |
|7369  |SMITH   |CLERK    |7902       |1980-04-04  |800       |NULL    |20      |
|7499  |ALLEN   |SALESMAN |7698       |1981-05-05  |1600      |300     |30      |
|7521  |WARD    |SALESMAN |7698       |1982-06-23  |1250      |500     |30      |
|7654  |MARTIN  

In [6]:

dept_columns = ['dept_id', 'dept_name', 'dept_location']

dept_schema = StructType() \
    .add("dept_id", IntegerType(), True) \
    .add("dept_name", StringType(), True) \
    .add("dept_location", StringType(), True)

dept_df = spark.read.format("csv") \
    .option("header", True) \
    .schema(dept_schema) \
    .load("file:///apps/sandbox/datasets/employee-data/departments.csv")

#dept_df.printSchema()

dept_df.show(truncate=False)

+-------+----------+-------------+
|dept_id|dept_name |dept_location|
+-------+----------+-------------+
|10     |ACCOUNTING|NEW YORK     |
|20     |RESEARCH  |BOSTON       |
|30     |SALES     |CHICAGO      |
|40     |OPERATIONS|BOSTON       |
|50     |ADMIN     |CHICAGO      |
+-------+----------+-------------+



### Spark Joins

#### Inner Join
Returns only the rows from both the dataframes that have matching values in both columns specified as the join keys.

```sql
df1.join(df2, df1['key'] == df2['key'], 'inner').show()
```

In [7]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "inner").show(truncate=False)

+------+--------+---------+-----------+------------+----------+--------+--------+-------+----------+-------------+
|emp_id|emp_name|emp_role |emp_manager|emp_hiredate|emp_salary|emp_comm|emp_dept|dept_id|dept_name |dept_location|
+------+--------+---------+-----------+------------+----------+--------+--------+-------+----------+-------------+
|7839  |KING    |PRESIDENT|NULL       |1981-11-17  |5000      |NULL    |10      |10     |ACCOUNTING|NEW YORK     |
|7698  |BLAKE   |MANAGER  |7839       |1981-01-05  |2850      |125     |30      |30     |SALES     |CHICAGO      |
|7782  |CLARK   |MANAGER  |7839       |1983-09-06  |2450      |NULL    |10      |10     |ACCOUNTING|NEW YORK     |
|7566  |JONES   |MANAGER  |7839       |1981-02-04  |2975      |345     |20      |20     |RESEARCH  |BOSTON       |
|7788  |SCOTT   |ANALYST  |7566       |1984-12-17  |3000      |NULL    |20      |20     |RESEARCH  |BOSTON       |
|7902  |FORD    |ANALYST  |7566       |1981-11-27  |3000      |NULL    |20      

#### Left / Left Outer Join
Returns all the rows from the left dataframe and the matching rows from the right dataframe. If there are no matching values in the right dataframe, then it returns a null.

`Syntax`
```sql
df1.join(df2, df1['key'] == df2['key'], 'left').show()
(OR)
df1.join(df2, df1['key'] == df2['key'], 'leftouter').show()
```

In [8]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "leftouter").show(truncate=False)

+------+--------+---------+-----------+------------+----------+--------+--------+-------+----------+-------------+
|emp_id|emp_name|emp_role |emp_manager|emp_hiredate|emp_salary|emp_comm|emp_dept|dept_id|dept_name |dept_location|
+------+--------+---------+-----------+------------+----------+--------+--------+-------+----------+-------------+
|7839  |KING    |PRESIDENT|NULL       |1981-11-17  |5000      |NULL    |10      |10     |ACCOUNTING|NEW YORK     |
|7698  |BLAKE   |MANAGER  |7839       |1981-01-05  |2850      |125     |30      |30     |SALES     |CHICAGO      |
|7782  |CLARK   |MANAGER  |7839       |1983-09-06  |2450      |NULL    |10      |10     |ACCOUNTING|NEW YORK     |
|7566  |JONES   |MANAGER  |7839       |1981-02-04  |2975      |345     |20      |20     |RESEARCH  |BOSTON       |
|7788  |SCOTT   |ANALYST  |7566       |1984-12-17  |3000      |NULL    |20      |20     |RESEARCH  |BOSTON       |
|7902  |FORD    |ANALYST  |7566       |1981-11-27  |3000      |NULL    |20      

#### Right / Right Outer Join
```
df1.join(df2, df1['key'] == df2['key'], 'right').show()
(OR)
df1.join(df2, df1['key'] == df2['key'], 'rightouter').show()
```

In [ ]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "right").show(truncate=False)

#### Outer / Full Join

```sql
df1.join(df2, df1['key'] == df2['key'], 'outer').show()
(OR)
df1.join(df2, df1['key'] == df2['key'], 'full').show()
(OR)
df1.join(df2, df1['key'] == df2['key'], 'fullouter').show()
```

In [ ]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "outer").show(truncate=False)

### Cross Join

```sql
df1.crossJoin(df2).show()
```

In [ ]:
#
spark.conf.set("spark.sql.crossJoin.enabled", True)

#
employee_df.crossJoin(dept_df).show(truncate=False)

#### Left Anti Join
A left anti join in Spark SQL is a type of left join operation that returns only the rows from the left dataframe that do not have matching values in the right dataframe. It is used to find the rows in one dataframe that do not have corresponding values in another dataframe.

```sql
df1.join(df2, df1['key'] == df2['key'], 'left_anti').show()
```

In [ ]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "left_anti").show(truncate=False)

#### Left Semi Join
A left semi join in Spark SQL is a type of join operation that returns only the columns from the left dataframe that have matching values in the right dataframe. It is used to find the values in one dataframe that have corresponding values in another dataframe.

```sql
df1.join(df2, df1['key'] == df2['key'], 'leftsemi').show()
```

In [ ]:
employee_df.join(dept_df, employee_df["emp_dept"] == dept_df["dept_id"], "leftsemi").show(truncate=False)

#### Self Join
A self join in Spark SQL is a join operation in which a dataframe is joined with itself. It is used to compare the values within a single dataframe and return the rows that match specified criteria.

```sql
df.alias("df1").join(df.alias("df2"), df1['key'] == df2['key']).show()
```

In [ ]:
employee_df.alias("employee_df1").join(employee_df.alias("employee_df2"), employee_df['emp_id'] == employee_df['emp_id']).show()